# CrossEntropyLoss : mesurer l’erreur de classification

Faire payer la préférence erronée pour B, sans mélanger plusieurs tâches.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#choisir-la-perte).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La couche produit une voix correcte pour A et C, mais préfère aussi la voix pour B. Nous avons besoin d’un coût qui distingue ces erreurs.

Nous gardons une seule classe attendue par extrait. scores, aussi appelé logits, est un tableau flottant (3,2). cibles, souvent nommé targets, contient trois indices entiers.

Dans ce mode, CrossEntropyLoss attend des indices de type long, alias de int64. La règle vient de ce contrat de perte, pas du mot cibles.

La formule prend la probabilité attribuée à la bonne classe et calcule −log(probabilité). Une valeur proche de un coûte peu ; une valeur proche de zéro coûte beaucoup.

Comment obtenir ces probabilités ? La softmax calcule exp(score) puis divise par la somme des exponentielles de la ligne. exp désigne la fonction exponentielle.

Pour A, la probabilité voix vaut exp(1,5)/(exp(1,5)+exp(−1,5)), soit environ 0,952574. Son coût −log(0,952574) vaut environ 0,048587.

| Extrait | Classe annotée | Probabilité de cette classe | Coût calculé |
| --- | --- | --- | --- |
| A | Voix 0 | 0,952574 | 0,048587 |
| B | Percussion 1 | 0,000123 | 9,000123 |
| C | Voix 0 | 0,993307 | 0,006715 |

Les valeurs sont arrondies à six décimales. B domine la moyenne parce que le modèle attribue presque toute sa préférence à la mauvaise classe.

Nous créons d’abord un objet critere, puis l’appelons sur scores et cibles. L’écriture CrossEntropyLoss()(scores,cibles) fusionne ces deux opérations ; nous la séparons ici.

## Démonstration 1 · Conserver une perte par extrait, puis moyenner

Relier chaque coût à la ligne qui l’a produit.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Les données et poids sont ceux du chapitre Linear. Leur répétition rend ce programme exécutable sans objets cachés.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
modele = torch.nn.Linear(2, 2)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
with torch.no_grad():
    modele.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.bias.zero_()


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
scores = modele(X)


In [8]:
observer('scores', scores)

scores forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]


### Construire la règle de perte

reduction="none" demande une perte par extrait au lieu d’une moyenne immédiate.

In [9]:
critere = torch.nn.CrossEntropyLoss(reduction="none")


### Calculer l’état suivant

critere(scores,cibles) utilise la bonne classe de chaque ligne. mean() moyenne ensuite les trois pertes.

In [10]:
pertes = critere(scores, cibles)
perte = pertes.mean()


In [11]:
observer('pertes', pertes)
observer('perte', perte)

pertes forme (3,) type torch.float32 requires_grad True
  valeurs : [0.04858732968568802, 9.000123023986816, 0.006715348921716213]
perte forme () type torch.float32 requires_grad True
  valeurs : 3.0184755325317383


### Calculer l’état suivant

softmax applique la formule du tableau. Nous l’utilisons pour expliquer ; la perte reçoit directement les scores bruts.

In [12]:
probabilites = scores.softmax(dim=1)


In [13]:
observer('probabilites', probabilites)

probabilites forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[0.9525741338729858, 0.04742587357759476], [0.9998766183853149, 0.00012339458044152707], [0.9933071732521057, 0.006692850962281227]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("probabilites", [[round(v, 6) for v in ligne] for ligne in probabilites.tolist()])
print("pertes", [round(v, 6) for v in pertes.tolist()], "forme", tuple(pertes.shape))
print("moyenne", round(perte.item(), 6), "forme", tuple(perte.shape))


probabilites [[0.952574, 0.047426], [0.999877, 0.000123], [0.993307, 0.006693]]
pertes [0.048587, 9.000123, 0.006715] forme (3,)
moyenne 3.018476 forme ()


### Calculer l’état suivant

tolist() fournit les nombres affichés, puis torch.tensor crée un tableau indépendant. Nous portons le score percussion de B à 5. Cette correction n’est pas un entraînement.

In [15]:
nouveaux_scores = torch.tensor(scores.tolist())
nouveaux_scores[1, 1] = 5.


In [16]:
observer('nouveaux_scores', nouveaux_scores)

nouveaux_scores forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, 5.0], [2.5, -2.5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [17]:
print("perte B corrigee", round(critere(nouveaux_scores, cibles)[1].item(), 6))

perte B corrigee 0.474077


La perte donne une raison de modifier la règle : réduire la forte erreur de B. Une décision argmax seule ne montrerait pas l’intensité de cette mauvaise préférence.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Quelle ligne reçoit la plus grande perte individuelle : 0, 1 ou 2 ?

In [18]:
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_cibles = torch.tensor([0,1,0])
exo_pertes = torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles)
exo_indice = ...  # Remplacer les points par votre réponse.
if exo_indice is Ellipsis or exo_indice is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_indice == exo_pertes.argmax().item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La seconde ligne préfère fortement la classe voix alors que la cible demande percussion.

</details>

### Exercice 2

Avec reduction="none", quelle est la forme de la sortie pour ces trois extraits ?

In [19]:
exo_scores = torch.zeros(3,2)
exo_cibles = torch.tensor([0,1,0])
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Une perte par extrait reste disponible avant une éventuelle moyenne.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_cibles = torch.tensor([0,1,0])
exo_pertes = torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles)
exo_indice = 1
if exo_indice is Ellipsis or exo_indice is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_indice == exo_pertes.argmax().item()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_scores = torch.zeros(3,2)
exo_cibles = torch.tensor([0,1,0])
exo_forme = (3,)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.CrossEntropyLoss(reduction="none")(exo_scores,exo_cibles).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [20]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()